# Day 3 — 비동기 처리와 에러 핸들링

### 2.2 실습: 동기 vs 비동기 실행 시간 비교

In [1]:
import nest_asyncio
nest_asyncio.apply()

import time
import asyncio

In [2]:
# 동기 방식: 순차 실행
def sync_task(name, seconds):
    print(f"  [{name}] 시작")
    time.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

print("===== 동기 실행 =====")
start = time.time()
sync_task("작업A", 2)
sync_task("작업B", 2)
sync_task("작업C", 2)
print(f"\n총 소요 시간: {time.time() - start:.1f}초")

===== 동기 실행 =====
  [작업A] 시작
  [작업A] 완료 (2초)
  [작업B] 시작
  [작업B] 완료 (2초)
  [작업C] 시작
  [작업C] 완료 (2초)

총 소요 시간: 6.0초


In [3]:
# 비동기 방식: 동시 실행
async def async_task(name, seconds):
    print(f"  [{name}] 시작")
    await asyncio.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

async def run_async():
    print("===== 비동기 실행 =====")
    start = time.time()
    await asyncio.gather(
        async_task("작업A", 2),
        async_task("작업B", 2),
        async_task("작업C", 2),
    )
    print(f"\n총 소요 시간: {time.time() - start:.1f}초")

await run_async()

===== 비동기 실행 =====
  [작업A] 시작
  [작업B] 시작
  [작업C] 시작
  [작업A] 완료 (2초)
  [작업B] 완료 (2초)
  [작업C] 완료 (2초)

총 소요 시간: 2.0초


### 3. 문제 시연: 동기 추론이 서버를 멈추는 순간

In [4]:
%%writefile app/main_sync_problem.py
"""
Day 3 - 섹션 3: 동기 추론의 문제점을 보여주는 서버
두 가지 버전의 엔드포인트를 비교합니다.
"""
import time
from fastapi import FastAPI

app = FastAPI(title="Sync vs Async Problem Demo")

INFERENCE_TIME = 3   # 추론에 3초 걸린다고 가정

# ===== 버전 1: async def 안에서 동기 작업 (문제 있음) =====
@app.post("/predict/blocking")
async def predict_blocking():
    """
    ⚠️ 문제 버전: async def 안에서 time.sleep (동기 블로킹)
    이벤트 루프가 멈추므로, 동시 요청을 처리할 수 없습니다.
    """
    time.sleep(INFERENCE_TIME)   # 동기 블로킹 — 이벤트 루프가 멈춤
    return {"result": "완료", "method": "blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI가 스레드풀에서 실행) =====
@app.post("/predict/threadpool")
def predict_threadpool():
    """
    일반 def: FastAPI가 자동으로 별도 스레드에서 실행합니다.
    이벤트 루프는 블로킹되지 않지만, 스레드풀 크기에 제한이 있습니다.
    """
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "method": "threadpool", "duration": INFERENCE_TIME}


# 헬스체크: 서버가 응답 가능한 상태인지 확인용
@app.get("/health")
async def health():
    return {"status": "healthy"}

Writing app/main_sync_problem.py


In [5]:
# 서버 실행
# ⚠️ 이전 섹션/Day에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" 에러 = 같은 포트에 서버가 이미 실행 중
#    해결: Kernel → Restart Kernel 후 이 셀부터 다시 실행

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_sync_problem:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

INFO:     Started server process [13260]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨


#### 3.2 실험 1: 동시 요청 시 blocking 엔드포인트의 문제

In [8]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_request(url, request_id):
    """단일 요청을 보내고 소요 시간을 측정합니다."""
    start = time.time()
    response = requests.post(url)
    elapsed = time.time() - start
    return {
        "request_id": request_id,
        "elapsed": round(elapsed, 1),
        "status": response.status_code,
    }

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보냅니다."""
    print(f"\n{'='*55}")
    print(f"  {n_requests}개 동시 요청 → {url}")
    print(f"{'='*55}")

    start = time.time()

    with ThreadPoolExecutor(max_workers=n_requests) as executor:
        futures = {
            executor.submit(send_request, url, i+1): i for i in range(n_requests)
        }

        results = []
        for future in as_completed(futures):
            results.append(future.result())

    total = time.time() - start

    # 결과 출력 (요청 ID순으로 정렬)
    for r in sorted(results, key=lambda x: x["request_id"]):
        print(f"  요청 #{r['request_id']}: {r['elapsed']}초")

    print(f"\n  전체 소요 시간: {round(total, 1)}초")
    return total

In [7]:
# 실험 1: blocking 엔드포인트 (async def + time.sleep)
total_blocking = concurrent_test("http://localhost:8000/predict/blocking", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/blocking
INFO:     127.0.0.1:3040 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:3041 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:3042 - "POST /predict/blocking HTTP/1.1" 200 OK
  요청 #1: 11.0초
  요청 #2: 5.0초
  요청 #3: 8.0초

  전체 소요 시간: 11.0초


In [8]:
# 실험 2: threadpool 엔드포인트 (일반 def)
total_threadpool = concurrent_test("http://localhost:8000/predict/threadpool", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/threadpool
INFO:     127.0.0.1:11954 - "POST /predict/threadpool HTTP/1.1" 200 OK
INFO:     127.0.0.1:11953 - "POST /predict/threadpool HTTP/1.1" 200 OK
INFO:     127.0.0.1:11952 - "POST /predict/threadpool HTTP/1.1" 200 OK
  요청 #1: 5.1초
  요청 #2: 5.1초
  요청 #3: 5.0초

  전체 소요 시간: 5.1초


#### 3.4 실험 3: blocking이 헬스체크까지 막는 현상

In [10]:
import threading

def test_health_during_inference(predict_url):
    """추론 중에 헬스체크가 응답하는지 테스트합니다."""

    results = {}

    def send_predict():
        start = time.time()
        requests.post(predict_url)
        results["predict"] = round(time.time() - start, 1)

    def send_health():
        time.sleep(0.5)   # 추론이 시작된 후 0.5초 뒤에 헬스체크
        start = time.time()
        resp = requests.get("http://localhost:8000/health")
        results["health"] = round(time.time() - start, 1)

    t1 = threading.Thread(target=send_predict)
    t2 = threading.Thread(target=send_health)
    t1.start()
    t2.start()
    t1.join()
    t2.join()

    return results

In [12]:
# blocking 버전
print("===== /predict/blocking 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/blocking")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 단순 상태 확인인데 2.5초 대기!")

print()

# threadpool 버전
print("===== /predict/threadpool 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/threadpool")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 즉시 응답!")

===== /predict/blocking 중 헬스체크 =====
INFO:     127.0.0.1:9186 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:9187 - "GET /health HTTP/1.1" 200 OK
  추론 응답: 5.0초
  헬스체크 응답: 4.5초    ← 단순 상태 확인인데 2.5초 대기!

===== /predict/threadpool 중 헬스체크 =====
INFO:     127.0.0.1:9206 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:9205 - "POST /predict/threadpool HTTP/1.1" 200 OK
  추론 응답: 5.1초
  헬스체크 응답: 2.1초    ← 즉시 응답!


### 4. 해결 패턴: run_in_executor로 블로킹 방지하기

In [13]:
import asyncio

# Before: 이벤트 루프를 막는 방식
async def predict_blocking():
    result = model(input_tensor)     # 이벤트 루프 멈춤
    return result

# After: 이벤트 루프를 막지 않는 방식
async def predict_non_blocking():
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,                        # None = 기본 스레드풀 사용
        model,                       # 실행할 함수
        input_tensor,                # 함수에 전달할 인자
    )
    return result

#### 4.2 실습: 세 가지 버전 비교

In [4]:
%%writefile app/main_async_solution.py
"""
Day 3 - 섹션 4: 세 가지 동시 처리 방식 비교
"""
import time
import asyncio
# from concmport FastAPI
from fastapi import FastAPI
from concurrent.futures import ThreadPoolExecutor, as_completed

app = FastAPI(title="Async Solution Demo")

INFERENCE_TIME = 3

# 커스텀 스레드풀 생성 (최대 4개 스레드)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")


def heavy_inference():
    """동기 함수: 모델 추론을 시뮬레이션합니다."""
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "duration": INFERENCE_TIME}


# ===== 버전 1: async def + 동기 작업 (문제 있음) =====
@app.post("/predict/v1-blocking")
async def predict_v1():
    """❌ 이벤트 루프를 막습니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v1-blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI 자동 스레드풀) =====
@app.post("/predict/v2-def")
def predict_v2():
    """⭕ FastAPI가 자동으로 별도 스레드에서 실행합니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v2-def", "duration": INFERENCE_TIME}


# ===== 버전 3: async def + run_in_executor (권장) =====
@app.post("/predict/v3-executor")
async def predict_v3():
    """✅ 명시적으로 스레드풀에 위임합니다."""
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        inference_executor,    # 커스텀 스레드풀 사용
        heavy_inference,       # 실행할 동기 함수
    )
    return {"method": "v3-executor", **result}


@app.get("/health")
async def health():
    return {"status": "healthy"}

Overwriting app/main_async_solution.py


In [5]:
# ⚠️ 섹션 3에서 서버를 실행했다면, 반드시 커널을 재시작하세요.

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_async_solution:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

INFO:     Started server process [12920]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨


In [6]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보내고 결과를 측정합니다."""
    def send(i):
        start = time.time()
        resp = requests.post(url)
        return {"id": i+1, "elapsed": round(time.time() - start, 1)}

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 1)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초")
    print(f"  전체: {total}초\n")
    return total

In [7]:
print("=" * 50)
print("버전 1: async def + time.sleep (blocking)")
print("=" * 50)
t1 = concurrent_test("http://localhost:8000/predict/v1-blocking")

버전 1: async def + time.sleep (blocking)
INFO:     127.0.0.1:9112 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:9113 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:9111 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
  요청 #1: 5.1초
  요청 #2: 8.1초
  요청 #3: 11.1초
  전체: 11.1초



In [9]:
print("=" * 50)
print("버전 2: 일반 def (FastAPI 자동 스레드풀)")
print("=" * 50)
t2 = concurrent_test("http://localhost:8000/predict/v2-def")

버전 2: 일반 def (FastAPI 자동 스레드풀)

  3개 동시 요청 → http://localhost:8000/predict/v2-def
INFO:     127.0.0.1:9274 - "POST /predict/v2-def HTTP/1.1" 200 OK
INFO:     127.0.0.1:9272 - "POST /predict/v2-def HTTP/1.1" 200 OK
INFO:     127.0.0.1:9273 - "POST /predict/v2-def HTTP/1.1" 200 OK
  요청 #1: 5.0초
  요청 #2: 5.0초
  요청 #3: 5.0초

  전체 소요 시간: 5.1초


In [10]:
print("=" * 50)
print("버전 3: async def + run_in_executor (권장)")
print("=" * 50)
t3 = concurrent_test("http://localhost:8000/predict/v3-executor")

버전 3: async def + run_in_executor (권장)

  3개 동시 요청 → http://localhost:8000/predict/v3-executor
INFO:     127.0.0.1:9288 - "POST /predict/v3-executor HTTP/1.1" 200 OK
INFO:     127.0.0.1:9287 - "POST /predict/v3-executor HTTP/1.1" 200 OK
INFO:     127.0.0.1:9289 - "POST /predict/v3-executor HTTP/1.1" 200 OK
  요청 #1: 5.1초
  요청 #2: 5.1초
  요청 #3: 5.1초

  전체 소요 시간: 5.1초


#### 4.4 Day 2 API에 적용하기

In [11]:
%%writefile app/main_v2.py
"""
Day 3 - Day 2 API에 비동기 패턴 적용
app/main.py의 개선 버전입니다.
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import (
    PixelPredictRequest,
    ImagePredictRequest,
    PredictResponse,
)
from app.model_utils import load_model, predict, preprocess


# ===== 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API (Async)",
    description="비동기 처리가 적용된 MNIST 추론 API",
    version="2.0.0",
)

# ===== 추론 전용 스레드풀 =====
inference_executor = ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="inference",
)

# ===== 모델 로드 =====
MODEL_PATH = "models/mnist_state_dict.pth"
model = load_model(MODEL_PATH)


# ===== 동기 추론 함수 (스레드풀에서 실행될 함수) =====
def run_inference(image_tensor: torch.Tensor) -> dict:
    """모델 추론을 수행합니다. 이 함수는 별도 스레드에서 실행됩니다."""
    return predict(model, image_tensor)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy", "model_loaded": model is not None}


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    """비동기 버전: 픽셀 배열로 추론"""
    try:
        # 전처리 (가벼운 작업 — 이벤트 루프에서 직접 수행)
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            pixel_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"추론 실패: {str(e)}")


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    """비동기 버전: Base64 이미지로 추론"""
    try:
        # 전처리 (가벼운 작업)
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            image_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

Writing app/main_v2.py


#### 4.5 스레드풀 크기 가이드

In [12]:
import os

# CPU 코어 수 확인
cpu_count = os.cpu_count()
print(f"CPU 코어 수: {cpu_count}")
print(f"권장 max_workers (CPU 추론): {cpu_count}")
print(f"권장 max_workers (GPU 추론): 1~2")

CPU 코어 수: 24
권장 max_workers (CPU 추론): 24
권장 max_workers (GPU 추론): 1~2


## 5. 에러 핸들링과 로깅

### 5.2 글로벌 Exception Handler

In [13]:
%%writefile app/error_handlers.py
"""
Day 3 - 글로벌 에러 핸들러
"""
import traceback
import logging

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")


def register_error_handlers(app: FastAPI):
    """FastAPI 앱에 글로벌 에러 핸들러를 등록합니다."""

    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        """모든 예외를 잡아서 안전한 응답을 반환합니다."""
        logger.error(
            f"에러 발생: {type(exc).__name__}: {exc}\n"
            f"경로: {request.method} {request.url}\n"
            f"스택 트레이스:\n{traceback.format_exc()}"
        )
        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": "서버 내부 오류가 발생했습니다.",
            }
            # ⚠️ 클라이언트에게는 상세 정보를 노출하지 않습니다.
            # 상세 정보는 서버 로그에만 기록됩니다.
        )

Writing app/error_handlers.py


### 5.3 Python logging 설정

In [14]:
%%writefile app/logger_config.py
"""
Day 3 - 로깅 설정
"""
import logging
import sys


def setup_logger(name: str = "ml_api", level: str = "INFO") -> logging.Logger:
    """콘솔 로거를 설정합니다."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))

    if logger.handlers:
        return logger

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        fmt="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger

Writing app/logger_config.py


In [15]:
# 로거 테스트
from app.logger_config import setup_logger

logger = setup_logger("ml_api")

logger.info("서버가 시작되었습니다.")
logger.warning("GPU 메모리가 80%를 초과했습니다.")
logger.error("모델 추론 중 에러가 발생했습니다.")

2026-03-31 12:19:30 INFO     [ml_api] 서버가 시작되었습니다.
2026-03-31 12:19:30 WARNING  [ml_api] GPU 메모리가 80%를 초과했습니다.
2026-03-31 12:19:30 ERROR    [ml_api] 모델 추론 중 에러가 발생했습니다.


### 5.4 요청/응답 로깅 미들웨어

In [16]:
%%writefile app/middleware.py
"""
Day 3 - 요청/응답 로깅 미들웨어
모든 요청의 메서드, 경로, 응답 시간, 상태 코드를 자동 로깅합니다.
"""
import time
import logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")


class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        duration = round(time.time() - start_time, 3)

        log_message = (
            f"{request.method} {request.url.path} "
            f"→ {response.status_code} "
            f"({duration}s)"
        )

        if response.status_code >= 500:
            logger.error(log_message)
        elif response.status_code >= 400:
            logger.warning(log_message)
        else:
            logger.info(log_message)

        response.headers["X-Process-Time"] = str(duration)
        return response

Writing app/middleware.py


## 6. 실습: 최종 서버 + 동시 요청 테스트

### 모든 이전 코드

In [5]:
import os

# 필요한 폴더 생성
for d in ["app", "models", "data"]:
    os.makedirs(d, exist_ok=True)

# 1. app/model_utils.py (Day 1)
if not os.path.exists("app/model_utils.py"):
    print("⚠️ app/model_utils.py 없음 → 생성합니다.")
    with open("app/model_utils.py", "w", encoding="utf-8") as f:
        f.write('''import torch
import torch.nn as nn
from torchvision import transforms

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

CLASS_NAMES = [str(i) for i in range(10)]

def load_model(model_path, num_classes=10):
    model = SimpleClassifier(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def predict(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)[0]
        idx = probs.argmax().item()
        return {
            "predicted_class": CLASS_NAMES[idx],
            "confidence": round(probs[idx].item(), 4),
            "probabilities": {CLASS_NAMES[i]: round(probs[i].item(), 4) for i in range(len(CLASS_NAMES))},
        }
''')
    print("  ✅ app/model_utils.py 생성 완료")
else:
    print("✅ app/model_utils.py 있음")

# 2. app/schemas.py (Day 2)
if not os.path.exists("app/schemas.py"):
    print("⚠️ app/schemas.py 없음 → 생성합니다.")
    with open("app/schemas.py", "w", encoding="utf-8") as f:
        f.write('''from pydantic import BaseModel, Field, field_validator
from typing import Optional

class PixelPredictRequest(BaseModel):
    pixels: list[list[float]] = Field(..., description="28x28 픽셀 배열")
    return_probabilities: bool = Field(default=False)
    @field_validator("pixels")
    @classmethod
    def validate_pixels(cls, v):
        if len(v) != 28:
            raise ValueError(f"28행이어야 합니다. 현재: {len(v)}행")
        for i, row in enumerate(v):
            if len(row) != 28:
                raise ValueError(f"각 행은 28열이어야 합니다. {i}번째 행: {len(row)}열")
        return v

class ImagePredictRequest(BaseModel):
    image_base64: str = Field(..., min_length=1)
    return_probabilities: bool = Field(default=False)

class PredictResponse(BaseModel):
    success: bool = Field(description="성공 여부")
    predicted_class: str = Field(description="예측 숫자 (0~9)")
    confidence: float = Field(description="확신도", ge=0.0, le=1.0)
    probabilities: Optional[dict[str, float]] = Field(default=None)
''')
    print("  ✅ app/schemas.py 생성 완료")
else:
    print("✅ app/schemas.py 있음")

# 3. app/error_handlers.py (Day 3 섹션 5)
if not os.path.exists("app/error_handlers.py"):
    print("⚠️ app/error_handlers.py 없음 → 생성합니다.")
    with open("app/error_handlers.py", "w", encoding="utf-8") as f:
        f.write('''import traceback, logging
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")

def register_error_handlers(app: FastAPI):
    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        logger.error(f"에러: {type(exc).__name__}: {exc}\\n경로: {request.method} {request.url}\\n{traceback.format_exc()}")
        return JSONResponse(status_code=500, content={"success": False, "error": "서버 내부 오류가 발생했습니다."})
''')
    print("  ✅ app/error_handlers.py 생성 완료")
else:
    print("✅ app/error_handlers.py 있음")

# 4. app/logger_config.py (Day 3 섹션 5)
if not os.path.exists("app/logger_config.py"):
    print("⚠️ app/logger_config.py 없음 → 생성합니다.")
    with open("app/logger_config.py", "w", encoding="utf-8") as f:
        f.write('''import logging, sys

def setup_logger(name="ml_api", level="INFO"):
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))
    if logger.handlers:
        return logger
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)-8s [%(name)s] %(message)s", "%Y-%m-%d %H:%M:%S"))
    logger.addHandler(handler)
    return logger
''')
    print("  ✅ app/logger_config.py 생성 완료")
else:
    print("✅ app/logger_config.py 있음")

# 5. app/middleware.py (Day 3 섹션 5)
if not os.path.exists("app/middleware.py"):
    print("⚠️ app/middleware.py 없음 → 생성합니다.")
    with open("app/middleware.py", "w", encoding="utf-8") as f:
        f.write('''import time, logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")

class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        start = time.time()
        response = await call_next(request)
        duration = round(time.time() - start, 3)
        msg = f"{request.method} {request.url.path} -> {response.status_code} ({duration}s)"
        if response.status_code >= 500: logger.error(msg)
        elif response.status_code >= 400: logger.warning(msg)
        else: logger.info(msg)
        response.headers["X-Process-Time"] = str(duration)
        return response
''')
    print("  ✅ app/middleware.py 생성 완료")
else:
    print("✅ app/middleware.py 있음")

# 6. 모델 파일 (Day 1)
if not os.path.exists("models/mnist_state_dict.pth"):
    print("⚠️ 모델 파일 없음 → MNIST 모델을 학습하여 생성합니다. (약 1~2분 소요)")
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    from app.model_utils import SimpleClassifier

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_data = DataLoader(datasets.MNIST("data", train=True, download=True, transform=transform), batch_size=64, shuffle=True)

    model = SimpleClassifier(10)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        for batch_idx, (images, labels) in enumerate(train_data):
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            if (batch_idx+1) % 300 == 0:
                print(f"  Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_data)}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "models/mnist_state_dict.pth")
    print("  ✅ models/mnist_state_dict.pth 생성 완료")
else:
    print("✅ models/mnist_state_dict.pth 있음")

print("\n🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.")

⚠️ app/model_utils.py 없음 → 생성합니다.
  ✅ app/model_utils.py 생성 완료
⚠️ app/schemas.py 없음 → 생성합니다.
  ✅ app/schemas.py 생성 완료
⚠️ app/error_handlers.py 없음 → 생성합니다.
  ✅ app/error_handlers.py 생성 완료
⚠️ app/logger_config.py 없음 → 생성합니다.
  ✅ app/logger_config.py 생성 완료
⚠️ app/middleware.py 없음 → 생성합니다.
  ✅ app/middleware.py 생성 완료
✅ models/mnist_state_dict.pth 있음

🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.


###  최종 서버 코드 통합

In [3]:
%%writefile app/main_final.py
"""
Day 3 최종 버전 - 비동기 + 에러 핸들링 + 로깅
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import PixelPredictRequest, ImagePredictRequest, PredictResponse
from app.model_utils import load_model, predict, preprocess
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


logger = setup_logger("ml_api")

app = FastAPI(
    title="MNIST Prediction API",
    description="비동기 처리, 에러 핸들링, 로깅이 적용된 MNIST 추론 API",
    version="3.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")

MODEL_PATH = "models/mnist_state_dict.pth"
model = None


@app.on_event("startup")
async def startup():
    global model
    logger.info(f"모델 로드 중: {MODEL_PATH}")
    model = load_model(MODEL_PATH)
    logger.info("모델 로드 완료")


def run_inference(image_tensor: torch.Tensor) -> dict:
    """별도 스레드에서 실행되는 추론 함수"""
    if model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return predict(model, image_tensor)


@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if model is not None else "loading",
        "model_loaded": model is not None,
    }


@app.get("/model/info", tags=["System"])
async def model_info():
    from app.model_utils import CLASS_NAMES
    total_params = sum(p.numel() for p in model.parameters())
    return {
        "model_name": "SimpleClassifier",
        "model_path": MODEL_PATH,
        "num_classes": len(CLASS_NAMES),
        "classes": CLASS_NAMES,
        "total_parameters": total_params,
    }


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    try:
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"전처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, pixel_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    try:
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)
    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, image_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )

Overwriting app/main_final.py


In [1]:
# ⚠️ 이전 섹션에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel 후 이 셀부터 실행

# 이 노트의 일부는 Day 2 sextion 5에서 생성한 파일을 활용합니다

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_final:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨")

INFO:     Started server process [9084]
INFO:     Waiting for application startup.


2026-03-31 16:19:51 INFO     [ml_api] 모델 로드 중: models/mnist_state_dict.pth


INFO:ml_api:모델 로드 중: models/mnist_state_dict.pth


2026-03-31 16:19:51 INFO     [ml_api] 모델 로드 완료


INFO:ml_api:모델 로드 완료
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


...

INFO:     127.0.0.1:2249 - "GET /docs HTTP/1.1" 200 OK
2026-03-31 16:19:55 INFO     [ml_api] GET /openapi.json -> 200 (0.006s)


INFO:ml_api:GET /openapi.json -> 200 (0.006s)


INFO:     127.0.0.1:2249 - "GET /openapi.json HTTP/1.1" 200 OK
2026-03-31 16:22:41 INFO     [ml_api] GET /model/info -> 200 (0.002s)


INFO:ml_api:GET /model/info -> 200 (0.002s)


INFO:     127.0.0.1:3149 - "GET /model/info HTTP/1.1" 200 OK


#### 동시 요청 테스트

In [1]:
import requests
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from torchvision import datasets

test_dataset = datasets.MNIST(root="data", train=False, download=True)

def concurrent_pixel_test(n_requests=3):
    """실제 모델 추론으로 동시 요청을 테스트합니다."""
    def send(i):
        image, label = test_dataset[i % len(test_dataset)]
        pixels = (np.array(image) / 255.0).tolist()
        start = time.time()
        resp = requests.post(
            "http://localhost:8000/predict/pixels",
            json={"pixels": pixels},
            timeout=30,
        )
        return {
            "id": i + 1,
            "elapsed": round(time.time() - start, 2),
            "status": resp.status_code,
        }

    print(f"\n{'='*50}")
    print(f"  {n_requests}개 동시 요청 (실제 추론)")
    print(f"{'='*50}")

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 2)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
    print(f"  전체: {total}초")

In [2]:
# 동시 요청 수를 늘려가며 테스트
for n in [1, 2, 4, 8]:
    concurrent_pixel_test(n)
    time.sleep(1)


  1개 동시 요청 (실제 추론)


NameError: name 'time' is not defined

#### 에러 핸들링 동작 확인

In [4]:
print("=" * 50)
print("  에러 핸들링 테스트")
print("=" * 50)

# 정상 요청
image, label = test_dataset[0]
pixels = (np.array(image) / 255.0).tolist()
resp = requests.post("http://localhost:8000/predict/pixels", json={"pixels": pixels})
print(f"\n[정상 요청] 상태: {resp.status_code}, 예측: {resp.json()['predicted_class']}")

# 잘못된 픽셀 크기
resp = requests.post(
    "http://localhost:8000/predict/pixels",
    json={"pixels": [[0.0] * 14 for _ in range(14)]}
)
print(f"[잘못된 크기] 상태: {resp.status_code}")

# 잘못된 Base64
resp = requests.post(
    "http://localhost:8000/predict/image",
    json={"image_base64": "not_valid!!!"}
)
print(f"[잘못된 Base64] 상태: {resp.status_code}, 에러: {resp.json().get('detail', 'N/A')}")

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"[헬스체크] 상태: {resp.status_code}, 응답: {resp.json()}")

  에러 핸들링 테스트
2026-03-31 12:46:58 INFO     [ml_api] POST /predict/pixels -> 200 (0.003s)


INFO:ml_api:POST /predict/pixels -> 200 (0.003s)


INFO:     127.0.0.1:14227 - "POST /predict/pixels HTTP/1.1" 200 OK

[정상 요청] 상태: 200, 예측: 7
2026-03-31 12:47:00 WARNING  [ml_api] POST /predict/pixels -> 422 (0.003s)


INFO:     127.0.0.1:14231 - "POST /predict/pixels HTTP/1.1" 422 Unprocessable Entity
[잘못된 크기] 상태: 422
2026-03-31 12:47:02 WARNING  [ml_api] POST /predict/image -> 400 (0.123s)


INFO:     127.0.0.1:14234 - "POST /predict/image HTTP/1.1" 400 Bad Request
[잘못된 Base64] 상태: 400, 에러: 이미지 처리 실패: cannot identify image file <_io.BytesIO object at 0x0000017C8B136D90>
2026-03-31 12:47:04 INFO     [ml_api] GET /health -> 200 (0.001s)


INFO:ml_api:GET /health -> 200 (0.001s)


INFO:     127.0.0.1:14237 - "GET /health HTTP/1.1" 200 OK
[헬스체크] 상태: 200, 응답: {'status': 'healthy', 'model_loaded': True}
